In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# BƯỚC 1: ĐỌC DỮ LIỆU GỐC
# products.csv: bảng PRODUCT đã có sẵn (coi như đã "sạch", đáng tin cậy).
# epd_csv.csv : dữ liệu sản phẩm mới/bổ sung (Extra Product Data), có thể
#               lẫn sản phẩm đã tồn tại và sản phẩm thực sự mới, đồng thời
#               có thể chứa lỗi (category sai, giá sai, trùng lặp...).
products = pd.read_csv("products.csv")
epd = pd.read_csv("epd_csv.csv")

print("Kích thước products:", products.shape)
print("Kích thước epd:", epd.shape)

# Làm việc trên bản copy, không sửa trực tiếp epd gốc
epd_clean = epd.copy()

Kích thước products: (2412, 8)
Kích thước epd: (1000, 8)


In [ ]:
# BƯỚC 2: CHUẨN HÓA KIỂU DỮ LIỆU / TEXT

# Các cột text có thể bị dính khoảng trắng thừa ở đầu/cuối (do nhập liệu
# thủ công hoặc export lỗi) -> strip() để tránh việc "Áo" và "Áo " bị coi
# là 2 giá trị khác nhau khi group/merge/so sánh sau này.
text_cols = ['product_name', 'category', 'segment', 'size', 'color']
for c in text_cols:
    epd_clean[c] = epd_clean[c].str.strip()

# Ép các cột số về đúng kiểu numeric.
# errors='coerce': giá trị không parse được (vd: text lẫn vào cột số) sẽ
# thành NaN thay vì làm chương trình lỗi -> mình vẫn phát hiện được ở
# bước kiểm tra tiếp theo.
epd_clean['price'] = pd.to_numeric(epd_clean['price'], errors='coerce')
epd_clean['cogs'] = pd.to_numeric(epd_clean['cogs'], errors='coerce')

# Danh sách category hợp lệ, lấy từ bảng products đã "sạch"
# -> dùng để đối chiếu, phát hiện category lạ/sai trong epd.
VALID_CATEGORIES = products['category'].unique().tolist()

# Ngưỡng để phát hiện giá bất thường (giá lớn gấp đôi giá cao nhất trong
# products là dấu hiệu khả nghi, ví dụ do nhập nhầm đơn vị hoặc thừa số 0).
MAX_PRICE_HOP_LY = products['price'].max() * 2

print("Số category hợp lệ (tham chiếu từ products):", len(VALID_CATEGORIES))
print("Ngưỡng giá bất thường (MAX_PRICE_HOP_LY):", MAX_PRICE_HOP_LY)

Số category hợp lệ (tham chiếu từ products): 4
Ngưỡng giá bất thường (MAX_PRICE_HOP_LY): 81900.0


In [ ]:
# BƯỚC 3: KIỂM TRA NULL Ở product_id

# Quyết định: KHÔNG xóa các dòng bị null product_id ở bước này, vì đây có
# thể là sản phẩm mới chưa được cấp ID (sẽ cấp ID mới ở Bước 9), không
# phải lỗi cần loại bỏ.
print("Số dòng product_id bị null:", epd_clean['product_id'].isna().sum())

Số dòng product_id bị null: 2


In [ ]:
# BƯỚC 4: SỬA LỖI DANH MỤC (category) VÀ GIÁ (price)

# Cách làm: nối (merge) epd với products theo (product_name, size, color)
# để "tra cứu" lại category/price đúng từ bảng products đã sạch.

# Chỉ giữ những (product_name, size, color) trong products là DUY NHẤT
# (chỉ xuất hiện đúng 1 lần), để tránh lấy nhầm giá trị nếu 1 tên sản
# phẩm ứng với nhiều dòng products khác nhau (không biết chọn dòng nào).
dem = products.groupby(['product_name', 'size', 'color']).size()
ten_duy_nhat = dem[dem == 1].index   # danh sách (tên, size, màu) chỉ xuất hiện 1 lần

bang_tra_cuu = products.set_index(['product_name', 'size', 'color']).loc[ten_duy_nhat]
bang_tra_cuu = bang_tra_cuu[['category', 'price']].reset_index()
bang_tra_cuu.columns = ['product_name', 'size', 'color', 'category_dung', 'price_dung']

# Ghép bảng tra cứu vào epd_clean (how='left': giữ nguyên toàn bộ dòng epd,
# chỉ thêm cột category_dung/price_dung nếu tra cứu được).
epd_clean = epd_clean.merge(bang_tra_cuu, on=['product_name', 'size', 'color'], how='left')

# Sửa category: nếu đang bị null hoặc mang giá trị lỗi 'Unknown_999'
# thì thay bằng giá trị đúng tra cứu được.
loi_category = epd_clean['category'].isna() | (epd_clean['category'] == 'Unknown_999')
print("Số dòng category lỗi:", loi_category.sum())
epd_clean.loc[loi_category, 'category'] = epd_clean.loc[loi_category, 'category_dung']

# Sửa price: nếu null, âm, hoặc lớn bất thường (> MAX_PRICE_HOP_LY)
# thì thay bằng giá trị đúng tra cứu được.
loi_price = (epd_clean['price'].isna()) | (epd_clean['price'] < 0) | (epd_clean['price'] > MAX_PRICE_HOP_LY)
print("Số dòng price lỗi:", loi_price.sum())
epd_clean.loc[loi_price, 'price'] = epd_clean.loc[loi_price, 'price_dung']

# Xóa 2 cột phụ đã dùng xong (chỉ là cột trung gian để tra cứu)
epd_clean = epd_clean.drop(columns=['category_dung', 'price_dung'])

# Kiểm tra lại: còn bao nhiêu dòng lỗi KHÔNG sửa được (do không tra cứu
# được từ products, ví dụ tên sản phẩm hoàn toàn mới)
print("Còn lại category lỗi chưa sửa được:",
      (epd_clean['category'].isna() | (epd_clean['category'] == 'Unknown_999')).sum())
print("Còn lại price lỗi chưa sửa được:",
      (epd_clean['price'].isna() | (epd_clean['price'] < 0)).sum())

Số dòng category lỗi: 2
Số dòng price lỗi: 3
Còn lại category lỗi chưa sửa được: 0
Còn lại price lỗi chưa sửa được: 0


In [ ]:
# BƯỚC 5: KIỂM TRA cogs <= price

# Về nghiệp vụ, giá vốn (cogs) không nên lớn hơn giá bán (price)
# -> chỉ kiểm tra và báo số lượng, chưa sửa ở bước này để nhóm quyết định
# cách xử lý (có thể là lỗi nhập liệu, cần xác nhận thêm).
print("Số dòng cogs > price:", (epd_clean['cogs'] > epd_clean['price']).sum())

Số dòng cogs > price: 0


In [ ]:
# BƯỚC 6: XÓA DÒNG TRÙNG HOÀN TOÀN TRONG EPD

# So sánh trùng lặp theo các cột mô tả sản phẩm (KHÔNG tính product_id,
# vì product_id trong epd có thể null hoặc chưa đáng tin cậy ở bước này).
# Nếu toàn bộ các cột mô tả giống hệt nhau -> chắc chắn là duplicate kỹ
# thuật (technical duplicate), giữ lại 1 dòng đầu tiên.
cac_cot_so_sanh = ['product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']

truoc = epd_clean.shape[0]
epd_clean = epd_clean.drop_duplicates(subset=cac_cot_so_sanh, keep='first')
print("Đã xóa", truoc - epd_clean.shape[0], "dòng trùng hoàn toàn trong EPD")

Đã xóa 0 dòng trùng hoàn toàn trong EPD


In [ ]:
# BƯỚC 7: SO SÁNH EPD VỚI PRODUCTS ĐỂ TÌM "BUSINESS DUPLICATE"

# "Business duplicate": sản phẩm trong epd thực ra đã tồn tại sẵn trong
# products (chỉ là được nhập lại), nên KHÔNG được coi là sản phẩm mới.

# LƯU Ý: KHÔNG dùng round() rồi so sánh bằng tuyệt đối! Những giá trị nằm
# đúng ranh giới làm tròn (ví dụ 5500.215) có thể round() ra kết quả KHÁC
# NHAU trên các máy/phiên bản pandas khác nhau (đã gặp lỗi này thực tế).
# Cách xử lý: so sánh bằng sai số cho phép (epsilon) thay vì bằng tuyệt đối.
EPSILON = 0.01   # sai số cho phép khi so sánh giá (< 1 xu là coi như bằng nhau)

cac_cot_phan_loai = ['product_name', 'category', 'segment', 'size', 'color']
epd_clean = epd_clean.reset_index(drop=True)
epd_clean['idx_goc'] = epd_clean.index   # đánh dấu vị trí gốc trước khi merge

# Ghép theo các cột TEXT trước (khớp chính xác 100%, không liên quan việc làm tròn số).
merged = epd_clean.merge(
    products[cac_cot_phan_loai + ['price', 'cogs']],
    on=cac_cot_phan_loai,
    how='left',
    suffixes=('', '_prod')
)

# 1 dòng epd có thể khớp text với NHIỀU dòng products (ví dụ cùng tên
# nhưng giá khác nhau) -> kiểm tra từng cặp, xem giá có GẦN NHAU không.
merged['gia_gan_nhau'] = (
    merged['price_prod'].notna() &
    (abs(merged['price'] - merged['price_prod']) < EPSILON) &
    (abs(merged['cogs']  - merged['cogs_prod'])  < EPSILON)
)

# Gom lại theo từng dòng epd gốc: chỉ cần khớp với ÍT NHẤT 1 dòng trong
# products (cả text lẫn giá) là đủ để coi là business duplicate.
ket_qua = merged.groupby('idx_goc')['gia_gan_nhau'].any()
la_business_dup = epd_clean['idx_goc'].map(ket_qua).values

print("Số dòng là business duplicate (đã có sẵn trong Products):", la_business_dup.sum())
print("Số dòng là sản phẩm mới thực sự:", (~la_business_dup).sum())

Số dòng là business duplicate (đã có sẵn trong Products): 1000
Số dòng là sản phẩm mới thực sự: 0


In [ ]:
# BƯỚC 8: GIỮ LẠI CÁC DÒNG SẢN PHẨM MỚI THỰC SỰ

# Loại bỏ toàn bộ business duplicate tìm được ở Bước 7.
# (Không cần quan tâm product_id gốc trong epd có trùng với products hay
#  không nữa, vì đã xác định qua thuộc tính rằng đây là sản phẩm khác
#  nhau -> giữ lại.)
epd_san_pham_moi = epd_clean[~la_business_dup].drop(columns=['idx_goc']).copy()
print("Sản phẩm mới còn lại:", epd_san_pham_moi.shape[0])

Sản phẩm mới còn lại: 0


In [ ]:
# BƯỚC 9: CẤP ID MỚI CHO CÁC SẢN PHẨM MỚI

# product_id trong epd_san_pham_moi có thể null/không đáng tin cậy
# (xem Bước 3), nên cấp lại ID mới, nối tiếp từ ID lớn nhất đang có
# trong products -> đảm bảo không trùng với ID đã tồn tại.
id_lon_nhat = products['product_id'].max()
so_luong_moi = epd_san_pham_moi.shape[0]

epd_san_pham_moi = epd_san_pham_moi.reset_index(drop=True)
epd_san_pham_moi['product_id'] = range(int(id_lon_nhat) + 1, int(id_lon_nhat) + 1 + so_luong_moi)

print("ID lớn nhất hiện có:", id_lon_nhat)
print("Đã cấp ID mới cho", so_luong_moi, "sản phẩm, từ", int(id_lon_nhat) + 1,
      "đến", int(id_lon_nhat) + so_luong_moi)

ID lớn nhất hiện có: 2412
Đã cấp ID mới cho 0 sản phẩm, từ 2413 đến 2412


In [ ]:
# BƯỚC 10: GỘP (CONCAT) VỚI PRODUCTS THÀNH BẢNG SILVER

# Ghép bảng products gốc với các sản phẩm mới thực sự (đã có ID) thành
# bảng PRODUCT Silver cuối cùng.
cac_cot_cuoi = ['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']
silver_product = pd.concat([products[cac_cot_cuoi], epd_san_pham_moi[cac_cot_cuoi]], ignore_index=True)

print("Bảng PRODUCT silver cuối cùng:", silver_product.shape)

Bảng PRODUCT silver cuối cùng: (2412, 8)


In [ ]:
# BƯỚC 11: KIỂM TRA TỔNG THỂ BẢNG SILVER TRƯỚC KHI EXPORT

# Bước validation cuối cùng, đảm bảo bảng Silver đạt yêu cầu chất lượng
# tối thiểu: không trùng khóa chính, không còn null, không vi phạm
# ràng buộc nghiệp vụ (cogs <= price, price > 0).
print("Tổng số dòng:", silver_product.shape[0])
print("Số product_id bị trùng:", silver_product['product_id'].duplicated().sum())
print("Số ô bị null:", silver_product.isnull().sum().sum())
print("Số dòng cogs > price:", (silver_product['cogs'] > silver_product['price']).sum())
print("Số dòng price <= 0:", (silver_product['price'] <= 0).sum())

Tổng số dòng: 2412
Số product_id bị trùng: 0
Số ô bị null: 0
Số dòng cogs > price: 0
Số dòng price <= 0: 0


In [ ]:
# BƯỚC 12: XUẤT FILE CSV

# Sau khi đã kiểm tra đầy đủ (Bước 1-11), xuất bảng Silver ra file CSV để dùng cho các bước tiếp theo.
silver_product.to_csv("silver_product.csv", index=False)

Đã lưu file silver_product.csv
